# Avaliação automática da qualidade da aorta

Comparação percentual de ImageCAS, MM-WHS, OrCaScore e **Geral**, separada por
treino, validação e teste e pelas resoluções mid/high. O Geral soma os exames
dos bancos disponíveis; não é a média simples dos percentuais dos bancos.
“Adequada” significa ausência dos alertas automáticos, não acurácia anatômica.

Para ImageCAS, a análise usa **somente os resultados canônicos**, sem fontes
históricas alternativas. Lê CSVs existentes, sem executar segmentações ou alterar
resultados. A avaliação visual e o Dice externos estão em
[Avaliação externa da aorta](../results/external_aorta_assessment.ipynb).

As seções finais relacionam o diagnóstico com o resultado dos óstios no
ImageCAS e com o Dice da aorta no MM-WHS, sem recalcular segmentações.

In [ ]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.analysis.aorta_automatic_quality import (
    compare_aorta_resolutions,
    compare_mmwhs_aorta_dice,
    default_aorta_feedback_sources,
    load_aorta_feedback_sources,
    summarize_aorta_feedback_comparison,
    summarize_imagecas_aorta_ostia,
)
from utils.project.runtime.paths import find_repository_root

REPO_ROOT = find_repository_root(REPO_ROOT)
pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. De onde vêm os dados?

**ImageCAS atual:** exclusivamente dos CSVs
`output/segmentation/canonical/{mid_res,high_res}/{train,val,test}/current/numeric/results_<split>.csv`.
São 30 exames de treino, 270 de validação e 700 de teste por resolução.
`current` pode ser um link para um run ou uma consolidação com origem registrada
em `source_manifest.json`. A comparação lê apenas a referência apontada por
`current` para cada resolução e conjunto.

**MM-WHS e OrCaScore:** por padrão, os diagnósticos já salvos nos CSVs locais
`output/segmentation/runs/<resolução>/external_ccta/<banco>/<run>/numeric/results_all.csv`:

| Banco | Resolução | Run local |
|---|---|---|
| MM-WHS | mid | `2026-09-22_18-51-28` |
| MM-WHS | high | `2026-09-22_17-47-45` |
| OrCaScore | mid | `2026-09-19_10-23-50` |
| OrCaScore | high | `2026-09-22_19-12-41` |

Esses arquivos já contêm o feedback automático; não é preciso reexecutar os
exames externos para consultá-lo. A seção MM-WHS reutiliza Dice salvo de treino e aceita teste somente com
o protocolo corrigido da aorta isolada; o Dice legado de coração completo é bloqueado.
Os runs locais MM-WHS de setembro são distintos dos runs corrigidos de outubro
usados na avaliação quantitativa externa.

Se `EXTERNAL_CCTA_RESULTS_ROOT` ou `CCTA_RESULTS_ROOT` estiver definida, ela
seleciona explicitamente a árvore externa `mmwhs/` e `orcascore/`, mantendo os
runs MM-WHS de outubro. Para usar os locais nessa situação, defina
`EXTERNAL_RESULTS_ROOT = None` na configuração antes de construir `SOURCES`.
Uma fonte explícita ausente não é substituída silenciosamente por outro run.

O gráfico de cobertura distingue diagnóstico disponível, coluna não persistida,
arquivo ausente e subconjunto não aplicável. Validação existe somente no ImageCAS.
As barras percentuais usam os **registros presentes no CSV** e incluem feedback
vazio e dados insuficientes; exames sem registro são informados na cobertura.
No teste OrCaScore high há 39 registros de 40 esperados.

O Geral considera somente bancos aplicáveis com diagnósticos disponíveis.
Se algum banco aplicável estiver ausente ou sem a coluna de diagnóstico,
o agregado será **Geral parcial**, com os bancos incluídos identificados na tabela.
Na validação, o Geral corresponde apenas ao ImageCAS. Ausência de diagnóstico
não é tratada como 0% de aortas adequadas. Nenhum subconjunto ou resolução é
somado a outro. Diferenças entre N presente e esperado permanecem informadas.

In [ ]:
external_root_setting = os.environ.get("EXTERNAL_CCTA_RESULTS_ROOT") or os.environ.get(
    "CCTA_RESULTS_ROOT"
)
EXTERNAL_RESULTS_ROOT = (
    Path(external_root_setting).expanduser() if external_root_setting else None
)
SOURCES = default_aorta_feedback_sources(REPO_ROOT, EXTERNAL_RESULTS_ROOT)
records, coverage = load_aorta_feedback_sources(SOURCES)
summary, comparison_coverage = summarize_aorta_feedback_comparison(records, coverage)

# Corte descritivo, sem ajuste a partir dos resultados de teste.
AORTA_DICE_THRESHOLD = 0.70

In [ ]:
from eda.aorta.view.aorta_automatic_quality_views import (
    show_absolute_relative_table,
    show_dataset_comparison,
    show_paired_transitions,
    show_source_coverage,
)

In [ ]:
show_source_coverage(coverage.loc[coverage["reference"].eq("Atual")])

## 2. Critérios automáticos da aorta

Os diagnósticos persistidos são produzidos por `classify_aorta_segmentation_feedback`, com os critérios de `LEVEL_SET.quality_feedback` dos respectivos runs. As referências selecionadas usam as regras abaixo. A classificação segue esta ordem:

1. **Dados insuficientes:** alguma das três medições — preenchimento dos círculos Q25, razão área/círculo P90 ou fração volumétrica da aorta — está ausente ou não é finita.
2. **Suspeita de sobresegmentação:** razão área/círculo P90 **> 3,0**; ou fração volumétrica **> 0,019** junto com razão área/círculo P90 **> 2,8**.
3. **Suspeita de subsegmentação:** pelo menos **2 dos 3 sinais**: preenchimento dos círculos Q25 **< 0,80**; razão área/círculo P90 **< 1,30**; fração volumétrica **< 0,008**.
4. **Adequada:** nenhuma das condições anteriores foi atendida.

Os limiares são estritos (maior/menor, sem incluir igualdade). “Adequada” significa ausência desses alertas, não confirmação por ground truth. A fração volumétrica depende do volume processado e do campo de visão; os alertas não substituem o Dice nem a inspeção visual.

**Sem feedback** é um registro com a coluna de diagnóstico presente, mas valor vazio.
**Diagnóstico não persistido** significa que a coluna inteira está ausente; não é uma classe do algoritmo.
**Arquivo ausente** e **Não se aplica** são estados da cobertura, não resultados de classificação.
Se um CSV canônico não tiver a coluna de diagnóstico, essa coorte ficará indisponível nesta análise até a promoção de uma referência com os diagnósticos persistidos.

## 3. Resultados relativos dos bancos e Geral

Cada painel compara ImageCAS, MM-WHS, OrCaScore e Geral no mesmo subconjunto e
resolução. As barras mostram as cinco categorias em uma escala comum de 0 a 100%,
com N ao lado do nome. O Geral, destacado por contorno, é ponderado pelo número
de exames: **soma das contagens da categoria / soma dos registros incluídos**.
O maior banco contribui com mais exames para o Geral, mas cada barra individual
tem o próprio denominador. Ausências aparecem com indicação textual.
“Geral parcial” considera somente os bancos identificados na tabela abaixo.

In [ ]:
current_summary = summary.loc[summary["reference"].eq("Atual")]
current_coverage = comparison_coverage.loc[comparison_coverage["reference"].eq("Atual")]
show_dataset_comparison(
    current_summary, current_coverage, "Qualidade automática da aorta — bancos e Geral"
)

## 4. Resultados absolutos e relativos

Cada categoria é apresentada como **n (percentual)**, usando o mesmo denominador
do gráfico. N por banco conta os registros lidos; N do Geral conta somente os
registros que contribuíram para o agregado. Uma coluna de diagnóstico ausente
aparece como “—”, não como zero. “Sem feedback” é uma categoria registrada e
permanece no denominador. N esperado permite identificar registros ausentes.

O Geral fica indisponível se nenhum banco contribuir. Na validação, os bancos
externos são “Não se aplica” e não tornam o agregado parcial.

In [ ]:
show_absolute_relative_table(current_summary, current_coverage)

## 5. Mudanças de classificação entre mid e high

Os mapas de calor mostram a classificação mid nas linhas e high nas colunas.
Cada célula é o percentual do **total de exames pareados do painel**, de forma
que as células somam 100%. O pareamento usa os mesmos IDs, separados por dataset
e subconjunto; o título informa o N e quantos IDs são exclusivos de cada resolução.
Somente exames com a coluna de diagnóstico presente nas duas resoluções entram.
Feedback vazio continua sendo “Sem feedback”.

In [ ]:
paired, pairing_coverage = compare_aorta_resolutions(
    records.loc[records["reference"].eq("Atual")]
)
show_paired_transitions(paired, pairing_coverage)

In [ ]:
from eda.aorta.view.aorta_automatic_quality_views import (
    show_imagecas_ostia_comparison,
    show_mmwhs_dice_comparison,
)

## 6. ImageCAS: qualidade da aorta × resultado dos óstios

Somente resultados **canonical**, separados por resolução e subconjunto.
**Com alertas** reúne suspeitas de subsegmentação e sobresegmentação;
**Indeterminada** reúne dados insuficientes e diagnóstico ausente.

**Ambos corretos** é o critério estrito de interseção com a referência.
**Apenas toleráveis** exclui os estritos e usa a tolerância salva na execução
(7 mm nas fontes canonical atuais). **Total aceito** soma essas duas categorias.
As demais categorias distinguem óstios incorretos, não encontrados e não avaliados.
Cada percentual usa **todos os exames do grupo**, incluindo não avaliados.
N=0 não representa taxa de 0%: é um grupo sem casos.

A comparação descreve uma associação, sem estabelecer causalidade ou validar
anatomicamente o diagnóstico automático. Uma aorta “Adequada” pode ter falha
nos óstios; casos com alertas também podem ter acerto.

In [ ]:
ostia_quality_summary = summarize_imagecas_aorta_ostia(
    records.loc[records["reference"].eq("Atual")]
)
show_imagecas_ostia_comparison(ostia_quality_summary)

## 7. MM-WHS: diagnóstico automático × Dice da aorta

O corte **Dice ≥ 0,70** é exploratório e inclusivo; não define um padrão de
qualidade anatômica. `AORTA_DICE_THRESHOLD` permite mudar esse corte descritivo,
sem ajustar o pipeline ou escolher o corte com base no teste. Os pontos e
boxplots mostram também os valores contínuos, além da tabela por diagnóstico.

Cada par diagnóstico/Dice vem do **mesmo registro e run**. Treino usa o label 820
na grade processada. Teste só participa com o protocolo
`mmwhs_aorta_label820_1mm_wine` (aorta isolada, label 820, 1 mm);
protocolos legados, ausentes ou misturados são bloqueados. As fontes locais de
setembro contêm apenas o teste legado. Para analisar teste corrigido, selecione
explicitamente a raiz externa com os runs corrigidos e seu feedback correspondente.

Os percentuais do corte e a mediana usam somente **Dice válido e avaliado**.
Dice zero efetivamente avaliado permanece; ausência nunca vira zero. A tabela
informa N, N avaliado, excluídos, cobertura e protocolo. “Sem feedback” e dados
insuficientes continuam identificados quando têm Dice válido. Treino e teste
usam protocolos distintos e não são agregados. Estes CSVs não fornecem acurácia
quantitativa dos óstios do MM-WHS.

In [ ]:
mmwhs_dice_cases, mmwhs_dice_summary, mmwhs_dice_coverage = compare_mmwhs_aorta_dice(
    records.loc[records["reference"].eq("Atual")], threshold=AORTA_DICE_THRESHOLD
)
show_mmwhs_dice_comparison(
    mmwhs_dice_cases,
    mmwhs_dice_summary,
    mmwhs_dice_coverage,
    coverage.loc[coverage["reference"].eq("Atual")],
    AORTA_DICE_THRESHOLD,
)